In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("prof_component_extracted_Stockholm")

# ---------- Load ALL firm CSVs into one panel ----------
MIN_YEAR = 2005
MAX_YEAR = 2024

rows = []
for fp in sorted(DATA_DIR.glob("*.csv")):
    firm = fp.stem
    df = pd.read_csv(fp)
    if "Year" not in df.columns:
        df = pd.read_csv(fp, index_col=0).reset_index().rename(columns={"index": "Year"})

    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df = df.dropna(subset=["Year"]).copy()
    df["Year"] = df["Year"].astype(int)
    df = df[(df["Year"] >= MIN_YEAR) & (df["Year"] <= MAX_YEAR)].sort_values("Year")

    # Ensure firm column exists
    df["firm"] = firm

    rows.append(df)

panel = pd.concat(rows, ignore_index=True)

panel["BE"] = pd.to_numeric(panel["BE"], errors="coerce")
panel["MIB"] = pd.to_numeric(panel["MIB"], errors="coerce")
panel["DENOM"] = panel["BE"] + panel["MIB"]

panel = panel.replace([np.inf, -np.inf], np.nan)
panel = panel.dropna(subset=["DENOM"]).copy()

# keep only positive denominators
panel = panel[panel["DENOM"] > 0].copy()

print(f"Panel after filter: {len(panel):,} rows, {panel['firm'].nunique():,} firms.")

print("Columns in panel:", panel.columns.tolist())

# Ensure PROF exists
if "PROF" not in panel.columns:
    raise ValueError("Column 'PROF' not found. Make sure you've computed PROF and saved it in each CSV.")


ValueError: No objects to concatenate